# H1 at a fixed frequency offset

## tl;dr

- The previous Model A result is treated as **inconclusive, not as a refutation of H1**.
- This notebook tests **behavioural H1 only** (Model A): recovery at an exact lock versus matched controls.
- The primary contrast uses fixed controls at \(f_k\pm1\) Hz and compares it with the legacy adaptive control.
- The default mode is `SYNTHETIC_SMOKE`: it performs no Chronos inference and every output is **NON-REPORTABLE**.
- `FULL` collects real Chronos observations, fits all/live variants, and emits a verdict only after recovery, convergence, BFMI, and posterior-predictive gates pass.
- No full-run result is embedded or assumed here.

## 0. Reproducible runtime

The notebook targets the project's tested legacy inference API: **PyMC 5.28.5**
with **ArviZ 0.23.4**. The bootstrap below installs those exact versions only when
the active runtime differs. This avoids silently switching to the PyMC 6/DataTree
API on a Python 3.12 Colab runtime. `FULL` additionally requires PyArrow, Torch and
`chronos-forecasting`; the synthetic smoke does not load a checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/patchAliasing

import os

os.environ["H1_RUN_MODE"] = "FULL"
os.environ["H1_NUTS_SAMPLER"] = "numpyro"
os.environ["H1_ARTIFACT_ROOT"] = "/content/drive/MyDrive/patchaliasing_h1"
os.environ["H1_RESUME"] = "1"
os.environ["H1_FORCE_COLLECTION"] = "0"

[Errno 2] No such file or directory: '/content/patchAliasing'
/content


In [ ]:
import importlib.metadata
import importlib.util
import os
import subprocess
import sys

REQUESTED_MODE = os.environ.get("H1_RUN_MODE", "SYNTHETIC_SMOKE").strip().upper()
if REQUESTED_MODE not in {"SYNTHETIC_SMOKE", "FULL"}:
    raise ValueError("H1_RUN_MODE must be SYNTHETIC_SMOKE or FULL")

def distribution_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

required_specs = []
if distribution_version("pymc") != "5.28.5":
    required_specs.append("pymc==5.28.5")
if distribution_version("arviz") != "0.23.4":
    required_specs.append("arviz==0.23.4")
for module, distribution, spec in (
    ("matplotlib", "matplotlib", "matplotlib>=3.10"),
    ("pandas", "pandas", "pandas>=2.2"),
    ("scipy", "scipy", "scipy>=1.14"),
):
    if importlib.util.find_spec(module) is None:
        required_specs.append(spec)
if REQUESTED_MODE == "FULL":
    if importlib.util.find_spec("pyarrow") is None:
        required_specs.append("pyarrow>=18")
    if importlib.util.find_spec("torch") is None:
        required_specs.append("torch")
    if distribution_version("chronos-forecasting") is None:
        required_specs.append("chronos-forecasting>=2.2.2")
if required_specs:
    print("Installing reproducible H1 runtime:", required_specs)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *required_specs]
    )

# The Windows smoke environment has no C++ compiler. PyTensor then uses its
# supported Python fallback; FULL/Colab keeps the optimized default.
if os.name == "nt" and REQUESTED_MODE == "SYNTHETIC_SMOKE":
    os.environ.setdefault("PYTENSOR_FLAGS", "cxx=")

Installing reproducible H1 runtime: ['arviz==0.23.4', 'chronos-forecasting>=2.2.2']


In [ ]:
!find /content -maxdepth 7 -type f -name h1_offset_lib.py

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
IN_COLAB = "google.colab" in sys.modules

def find_repo_root() -> Path:
    starts = []
    if os.environ.get("PATCHALIASING_REPO"):
        starts.append(Path(os.environ["PATCHALIASING_REPO"]))
    starts.append(Path.cwd())
    for start in starts:
        start = start.resolve()
        for candidate in (start, *start.parents):
            if (candidate / "chronos" / "bayesian" / "h1_offset_lib.py").exists():
                return candidate

    if IN_COLAB:
        target = Path("/content/patchAliasing")
        if not target.exists():
            subprocess.check_call(
                ["git", "clone", "--depth", "1", REPO_URL, str(target)]
            )
        if (target / "chronos" / "bayesian" / "h1_offset_lib.py").exists():
            return target
    raise FileNotFoundError(
        "Repository root not found. Run inside patchAliasing, set "
        "PATCHALIASING_REPO, or make h1_offset_lib.py available on the cloned branch."
    )

REPO_ROOT = find_repo_root()
BAYES_DIR = REPO_ROOT / "chronos" / "bayesian"
if str(BAYES_DIR) not in sys.path:
    sys.path.insert(0, str(BAYES_DIR))

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from IPython.display import display
from scipy import stats

if pm.__version__ != "5.28.5" or az.__version__ != "0.23.4":
    raise RuntimeError(
        f"Unsupported Bayesian runtime: PyMC {pm.__version__}, ArviZ {az.__version__}. "
        "Re-run the preceding bootstrap cell and restart the kernel if packages changed."
    )

import probe_lib as pl
from h1_offset_lib import (
    FROZEN_BAYES_MODELS,
    H1OffsetConfig,
    collect_paired_offsets,
    design_fingerprint,
    select_common_sites,
    synthetic_paired_data,
    to_long_contrasts,
    validate_paired_data,
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

RUN_MODE = REQUESTED_MODE
SYNTHETIC_SMOKE = RUN_MODE == "SYNTHETIC_SMOKE"
FULL = RUN_MODE == "FULL"

print("=" * 78)
print(f"H1 FIXED-OFFSET ANALYSIS | {RUN_MODE}")
if SYNTHETIC_SMOKE:
    print("*** SYNTHETIC SMOKE: NON-REPORTABLE; NO CHRONOS RESULT IS PRODUCED ***")
print("=" * 78)
print(f"Python {sys.version.split()[0]} | PyMC {pm.__version__} | ArviZ {az.__version__}")

FileNotFoundError: Repository root not found. Run inside patchAliasing, set PATCHALIASING_REPO, or make h1_offset_lib.py available on the cloned branch.

## Context & Methods

The existing behavioural analysis does **not** use \(\pm0.25\) Hz. Its adaptive controls can
be several hertz away because their maximum distance is \(0.25f_s/S\) (about 8 Hz at the
median site in the previous run). This notebook asks a narrower question:

> Is recovery specifically worse at the exact lock than at its immediate \(\pm1\) Hz neighbours?

The fixed-offset estimand reduces confounding from broad reconstruction ceilings and global
frequency trends, but it is not automatically better: if degradation spans more than 1 Hz,
both controls can lie inside the same valley and the contrast will shrink. The adaptive result
is therefore retained on the very same signals, together with a paired offset-difference fit.

This is a **post-hoc sensitivity analysis**, not an independent replication: inference seed 42
and the historical background pool seeds 10000--10099 are deliberately reused to make the two
offsets comparable. Model B is intentionally left unchanged and is not run here.

### Key Assumptions and decision rule

- The helper freezes 21 geometries (including `p8-s5`, excluding `p32-s28`) and uses the same
  intersection of clean sites for both offsets.
- Lock and all four controls share checkpoint, generator, background and phase; `R_lock` is
  inferred exactly once per pair.
- As requested, `fixed_live` is the primary estimand and the complete `fixed_all` view is the
  mandatory filter sensitivity. `live` is response-conditioned, so a disagreement is reported
  explicitly as **not robust to the live filter** rather than hidden.
- `SUPPORTED` requires \(P(\bar\beta<\log(0.8)\mid D)\ge0.95\).
  `PRACTICALLY ABSENT` requires
  \(P(|\bar\beta|<\log(1.1)\mid D)\ge0.95\). Otherwise the result is `INCONCLUSIVE`.
- A full verdict is blocked unless every required fit has rank \(\hat R<1.01\), bulk and tail
  ESS \(>1000\), zero divergences, BFMI \(\ge0.30\), and at least 90% stratified PPC coverage.
- Synthetic smoke uses relaxed numerical thresholds only to exercise the pipeline. It can
  never produce a reportable H1 verdict.

In [ ]:
DEVICE = os.environ.get("H1_DEVICE") or None
default_sampler = "numpyro"
NUTS_SAMPLER = os.environ.get("H1_NUTS_SAMPLER", default_sampler).strip().lower()
if NUTS_SAMPLER not in {"pymc", "numpyro", "nutpie"}:
    raise ValueError("H1_NUTS_SAMPLER must be pymc, numpyro, or nutpie")
if NUTS_SAMPLER != "pymc":
    backend_distribution = {"numpyro": "numpyro", "nutpie": "nutpie"}[NUTS_SAMPLER]
    if importlib.util.find_spec(backend_distribution) is None:
        if importlib.util.find_spec("pip") is None:
            raise RuntimeError(
                f"{backend_distribution} is missing and this interpreter has no pip. "
                f"With uv, add --with {backend_distribution} to the documented command."
            )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", backend_distribution]
        )
    if NUTS_SAMPLER == "numpyro":
        import jax
        print("JAX devices:", jax.devices())

RESUME = os.environ.get("H1_RESUME", "1") != "0"
FORCE_COLLECTION = os.environ.get("H1_FORCE_COLLECTION", "0") == "1"

CONFIG = (
    H1OffsetConfig(
        n_phase=2,
        n_bg=2,
        batch_size=32,
        seed=SEED,
        smoke=True,
    )
    if SYNTHETIC_SMOKE
    else H1OffsetConfig(device=DEVICE, seed=SEED)
)
if CONFIG.seed != SEED or not np.isclose(CONFIG.fixed_offset_hz, 1.0):
    raise ValueError("This notebook is frozen to seed=42 and fixed_offset_hz=1.0")

# Data shards and Bayesian fits have deliberately separate identities. Changing a
# prior, sampler, diagnostic, or model version must never force 487,400 Chronos rows
# to be collected again when the helper's collection design is unchanged.
DATA_DESIGN_FINGERPRINT = design_fingerprint(
    CONFIG, FROZEN_BAYES_MODELS
)[:12]
MODEL_VERSION = "h1_zero_sum_nested_v3"
notebook_document = json.loads(
    (BAYES_DIR / "h1_fixed_offset_analysis.ipynb").read_text(encoding="utf-8")
)
model_prefixes = (
    "DEVICE = os.environ.get(\"H1_DEVICE\") or None",
    "NU = 4",
    "def build_h1_model(",
    "def dataframe_fingerprint(",
)
model_sources = []
for prefix in model_prefixes:
    matches = [
        item["source"]
        for item in notebook_document["cells"]
        if item.get("cell_type") == "code"
        and "".join(item.get("source", [])).startswith(prefix)
    ]
    if len(matches) != 1:
        raise RuntimeError(f"cannot fingerprint notebook model cell {prefix!r}")
    source = matches[0]
    model_sources.append("".join(source) if isinstance(source, list) else source)
MODEL_CODE_FINGERPRINT = hashlib.sha256(
    "\n\n".join(model_sources).encode("utf-8")
).hexdigest()[:16]

runtime_versions = {
    "python": sys.version.split()[0],
    "pymc": pm.__version__,
    "arviz": az.__version__,
    "nuts_sampler": NUTS_SAMPLER,
    "nuts_backend_version": (
        pm.__version__
        if NUTS_SAMPLER == "pymc"
        else distribution_version(NUTS_SAMPLER)
    ),
    "jax": distribution_version("jax") if NUTS_SAMPLER == "numpyro" else None,
}
config_payload = {
    "data_design_fingerprint": DATA_DESIGN_FINGERPRINT,
    "model_version": MODEL_VERSION,
    "model_code_fingerprint": MODEL_CODE_FINGERPRINT,
    "runtime_versions": runtime_versions,
}
config_json = json.dumps(config_payload, sort_keys=True, default=str)
CONFIG_FINGERPRINT = hashlib.sha256(config_json.encode()).hexdigest()[:12]

artifact_base = Path(
    os.environ.get("H1_ARTIFACT_ROOT", str(BAYES_DIR / "_run"))
).expanduser()
NAMESPACE_ROOT = artifact_base / "h1_fixed_offset"
DATA_DIR = NAMESPACE_ROOT / "data" / f"{RUN_MODE.lower()}_{DATA_DESIGN_FINGERPRINT}"
FIT_DIR = NAMESPACE_ROOT / "fits" / f"{RUN_MODE.lower()}_{CONFIG_FINGERPRINT}"
RESULT_DIR = NAMESPACE_ROOT / "results" / f"{RUN_MODE.lower()}_{CONFIG_FINGERPRINT}"
for directory in (DATA_DIR, FIT_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MAIN_SETTINGS = (
    dict(draws=20, tune=20, chains=2, target_accept=0.85)
    if SYNTHETIC_SMOKE
    else dict(draws=2000, tune=3000, chains=4, target_accept=0.95)
)
AUX_SETTINGS = (
    dict(draws=15, tune=15, chains=2, target_accept=0.85)
    if SYNTHETIC_SMOKE
    else dict(draws=1000, tune=1500, chains=4, target_accept=0.95)
)
GATE = (
    dict(max_rhat=1.20, min_ess=10, max_divergences=0, min_bfmi=0.20, min_ppc=0.50)
    if SYNTHETIC_SMOKE
    else dict(max_rhat=1.01, min_ess=1000, max_divergences=0, min_bfmi=0.30, min_ppc=0.90)
)

print(f"data design fingerprint: {DATA_DESIGN_FINGERPRINT}")
print(f"fit config fingerprint: {CONFIG_FINGERPRINT}")
print(f"model code fingerprint: {MODEL_CODE_FINGERPRINT}")
print(f"frozen geometries: {len(FROZEN_BAYES_MODELS)}")
print(f"artifact namespace: {NAMESPACE_ROOT}")
print(f"sampling: {MAIN_SETTINGS} | backend={NUTS_SAMPLER}")

### Identifiable H1-only hierarchy

For each contrast \(d_i\),

\[
d_i\sim t_4(\mu_i,\sigma),\qquad
\mu_i=\bar\beta+
u^{cfg}_{c[i]}+u^{site}_{s[i]}+u^{bg}_{b[i]}+
u^{gen}_{g[i]}+u^{phase}_{p[i]}.
\]

Every effect is represented in an orthonormal zero-sum basis. Site effects sum to zero **within configuration**, background effects within generator, and `site#phase_idx` effects within site. These constraints prevent the intercept and group offsets from trading arbitrary constants. No overlap or patch-size slope is included: this notebook answers H1, while geometry heterogeneity is shown directly.

In [ ]:
NU = 4
PRIMARY_PRIOR_SCALE = 0.5
PRIOR_LADDER = (0.25, 0.5, 1.0)
ATTENUATION_20 = float(np.log(0.8))
ROPE_LOG = float(np.log(1.1))

def helmert_basis(n_levels: int) -> np.ndarray:
    if n_levels < 2:
        return np.zeros((n_levels, 0), dtype=float)
    basis = np.zeros((n_levels, n_levels - 1), dtype=float)
    for column in range(n_levels - 1):
        denom = np.sqrt((column + 1) * (column + 2))
        basis[: column + 1, column] = 1.0 / denom
        basis[column + 1, column] = -(column + 1) / denom
    if not np.allclose(basis.sum(axis=0), 0.0):
        raise AssertionError("zero-sum basis construction failed")
    return basis

def nested_zero_sum_basis(parent_by_level: list[str]) -> np.ndarray:
    parents = np.asarray(parent_by_level, dtype=str)
    blocks = []
    for parent in sorted(set(parents)):
        indices = np.flatnonzero(parents == parent)
        blocks.append((indices, helmert_basis(len(indices))))
    n_dof = sum(block.shape[1] for _, block in blocks)
    basis = np.zeros((len(parents), n_dof), dtype=float)
    start = 0
    for indices, block in blocks:
        stop = start + block.shape[1]
        if stop > start:
            basis[np.ix_(indices, np.arange(start, stop))] = block
        start = stop
    return basis

def factorize(values: pd.Series) -> tuple[np.ndarray, list[str]]:
    codes, levels = pd.factorize(values.astype(str), sort=True)
    if np.any(codes < 0):
        raise ValueError("missing factor level")
    return codes.astype("int64"), list(map(str, levels))

def prepare_model_frame(frame: pd.DataFrame) -> dict:
    required = {
        "pair_key", "model", "generator", "bg_id", "f_lock", "family",
        "phase_idx", "d", "live",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"model frame is missing columns: {sorted(missing)}")

    compact_columns = [
        column for column in (
            "pair_key", "data_origin", "reportable", "offset_kind", "delta_hz",
            "model", "generator", "bg_id", "f_lock", "family", "phase_idx",
            "d", "live",
        ) if column in frame.columns
    ]
    data = (
        frame.dropna(subset=["d"])
        .loc[:, compact_columns]
        .copy()
        .reset_index(drop=True)
    )
    if data.empty:
        raise ValueError("model frame has no observations")
    data["site"] = data["model"].astype(str) + "|" + data["f_lock"].map(lambda x: f"{x:.6f}")
    data["background"] = data["generator"].astype(str) + "#" + data["bg_id"].astype(str)
    data["phase_unit"] = data["site"] + "#phase" + data["phase_idx"].astype(str)

    codes, levels = {}, {}
    for name in ("model", "site", "generator", "background", "phase_unit"):
        codes[name], levels[name] = factorize(data[name])

    site_parent = (
        data[["site", "model"]].drop_duplicates().set_index("site")
        .reindex(levels["site"])["model"].astype(str).tolist()
    )
    background_parent = (
        data[["background", "generator"]].drop_duplicates().set_index("background")
        .reindex(levels["background"])["generator"].astype(str).tolist()
    )
    phase_parent = (
        data[["phase_unit", "site"]].drop_duplicates().set_index("phase_unit")
        .reindex(levels["phase_unit"])["site"].astype(str).tolist()
    )

    bases = {
        "config": helmert_basis(len(levels["model"])),
        "site": nested_zero_sum_basis(site_parent),
        "generator": helmert_basis(len(levels["generator"])),
        "background": nested_zero_sum_basis(background_parent),
        "phase": nested_zero_sum_basis(phase_parent),
    }
    model_frame = data.drop(columns=["site", "background", "phase_unit"])
    return {
        "frame": model_frame,
        "y": data["d"].to_numpy(float),
        "codes": {
            "config": codes["model"], "site": codes["site"],
            "generator": codes["generator"], "background": codes["background"],
            "phase": codes["phase_unit"],
        },
        "levels": {
            "config": levels["model"], "site": levels["site"],
            "generator": levels["generator"], "background": levels["background"],
            "phase": levels["phase_unit"],
        },
        "bases": bases,
    }

In [ ]:
def build_h1_model(
    prepared: dict,
    *,
    prior_scale: float = PRIMARY_PRIOR_SCALE,
    likelihood: str = "student",
) -> pm.Model:
    if likelihood not in {"student", "normal"}:
        raise ValueError("likelihood must be student or normal")

    coords = {"obs_id": np.arange(len(prepared["y"]))}
    for name, levels in prepared["levels"].items():
        coords[name] = levels
        n_dof = prepared["bases"][name].shape[1]
        if n_dof:
            coords[f"{name}_dof"] = np.arange(n_dof)

    with pm.Model(coords=coords) as model:
        beta_bar = pm.StudentT("beta_bar", nu=NU, mu=0.0, sigma=prior_scale)

        effects = {}
        for name in ("config", "site", "background", "generator", "phase"):
            basis = prepared["bases"][name]
            if basis.shape[1] == 0:
                effects[name] = pm.Deterministic(
                    f"u_{name}", pt.zeros((basis.shape[0],)), dims=name
                )
                continue
            effect_scale = pm.HalfStudentT(
                f"sigma_{name}", nu=NU, sigma=prior_scale
            )
            raw = pm.Normal(f"z_{name}", 0.0, 1.0, dims=f"{name}_dof")
            effects[name] = pm.Deterministic(
                f"u_{name}",
                effect_scale * pt.dot(pt.as_tensor_variable(basis), raw),
                dims=name,
            )

        mu = beta_bar
        for name in ("config", "site", "background", "generator", "phase"):
            mu = mu + effects[name][prepared["codes"][name]]

        sigma = pm.HalfStudentT("sigma", nu=NU, sigma=prior_scale)
        if likelihood == "student":
            pm.StudentT("d", nu=NU, mu=mu, sigma=sigma,
                        observed=prepared["y"], dims="obs_id")
        else:
            pm.Normal("d", mu=mu, sigma=sigma,
                      observed=prepared["y"], dims="obs_id")
        pm.Deterministic("recovery_ratio", pm.math.exp(beta_bar))
    return model

print("H1-only zero-sum model factory ready")

In [ ]:
def dataframe_fingerprint(
    frame: pd.DataFrame,
    *,
    prior_scale: float,
    likelihood: str,
    settings: dict,
) -> str:
    columns = [
        column for column in (
            "pair_key", "offset_kind", "model", "generator", "bg_id",
            "f_lock", "phase_idx", "d", "live",
        ) if column in frame.columns
    ]
    ordered = frame[columns].sort_values(columns[:-2] or columns).reset_index(drop=True)
    row_hash = pd.util.hash_pandas_object(ordered, index=False).values.tobytes()
    payload = json.dumps(
        {
            "config_fingerprint": CONFIG_FINGERPRINT,
            "model_version": MODEL_VERSION,
            "model_code_fingerprint": MODEL_CODE_FINGERPRINT,
            "nuts_sampler": NUTS_SAMPLER,
            "runtime_versions": runtime_versions,
            "prior_scale": prior_scale,
            "likelihood": likelihood,
            "settings": settings,
        },
        sort_keys=True,
    ).encode()
    return hashlib.sha256(payload + row_hash).hexdigest()[:16]

def fit_one(
    label: str,
    frame: pd.DataFrame,
    *,
    prior_scale: float = PRIMARY_PRIOR_SCALE,
    likelihood: str = "student",
    settings: dict | None = None,
) -> dict:
    settings = dict(settings or MAIN_SETTINGS)
    prepared = prepare_model_frame(frame)
    fingerprint = dataframe_fingerprint(
        prepared["frame"], prior_scale=prior_scale,
        likelihood=likelihood, settings=settings,
    )
    safe_label = "".join(char if char.isalnum() or char in "-_" else "_" for char in label)
    path = FIT_DIR / f"{safe_label}__{fingerprint}.nc"

    if RESUME and path.exists():
        print(f"checkpoint <- {path.name}")
        try:
            idata = az.from_netcdf(path)
        except Exception as error:
            raise RuntimeError(
                f"cached fit {path.name} is unreadable; rerun with H1_RESUME=0 "
                "or remove only that fingerprinted file"
            ) from error
    else:
        model = build_h1_model(
            prepared, prior_scale=prior_scale, likelihood=likelihood
        )
        sample_kwargs = dict(
            **settings,
            cores=1,
            random_seed=SEED,
            progressbar=not SYNTHETIC_SMOKE,
        )
        if NUTS_SAMPLER != "pymc":
            sample_kwargs["nuts_sampler"] = NUTS_SAMPLER
            if NUTS_SAMPLER == "numpyro" and SYNTHETIC_SMOKE:
                sample_kwargs["nuts_sampler_kwargs"] = {
                    "chain_method": "vectorized",
                    "nuts_kwargs": {"max_tree_depth": 4},
                }
        elif SYNTHETIC_SMOKE:
            sample_kwargs["nuts"] = {"max_treedepth": 4}
        with model:
            idata = pm.sample(**sample_kwargs)
        idata.attrs.update({
            "label": label,
            "fingerprint": fingerprint,
            "config_fingerprint": CONFIG_FINGERPRINT,
            "model_version": MODEL_VERSION,
            "model_code_fingerprint": MODEL_CODE_FINGERPRINT,
            "nuts_sampler": NUTS_SAMPLER,
            "runtime_versions": json.dumps(runtime_versions, sort_keys=True),
            "run_mode": RUN_MODE,
            "reportable": "true" if FULL else "false",
        })
        temporary = path.with_name(f".{path.stem}.tmp{path.suffix}")
        try:
            idata.to_netcdf(temporary)
            temporary.replace(path)
        finally:
            if temporary.exists():
                temporary.unlink()
        print(f"checkpoint -> {path.name}")

    expected_attrs = {
        "fingerprint": fingerprint,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "model_version": MODEL_VERSION,
        "model_code_fingerprint": MODEL_CODE_FINGERPRINT,
        "nuts_sampler": NUTS_SAMPLER,
        "runtime_versions": json.dumps(runtime_versions, sort_keys=True),
        "run_mode": RUN_MODE,
    }
    mismatches = {
        name: (idata.attrs.get(name), expected)
        for name, expected in expected_attrs.items()
        if str(idata.attrs.get(name, "")) != str(expected)
    }
    if mismatches:
        raise RuntimeError(f"checkpoint metadata mismatch for {label}: {mismatches}")
    return {
        "label": label, "idata": idata, "prepared": prepared,
        "fingerprint": fingerprint, "path": path,
        "prior_scale": prior_scale, "likelihood": likelihood,
    }

In [ ]:
def posterior_array(idata: az.InferenceData, variable: str) -> np.ndarray:
    values = idata.posterior[variable]
    other_dims = [dim for dim in values.dims if dim not in {"chain", "draw"}]
    return (
        values.stack(sample=("chain", "draw"))
        .transpose("sample", *other_dims)
        .values
    )

def posterior_predictive_strata(
    fit: dict,
    *,
    max_draws: int = 200,
    max_obs_per_stratum: int = 200,
) -> pd.DataFrame:
    idata, prepared = fit["idata"], fit["prepared"]
    frame, y, codes = prepared["frame"], prepared["y"], prepared["codes"]
    n_samples = idata.posterior.sizes["chain"] * idata.posterior.sizes["draw"]
    draw_index = np.linspace(
        0, n_samples - 1, min(max_draws, n_samples), dtype=int
    )

    beta = posterior_array(idata, "beta_bar")[draw_index]
    sigma = posterior_array(idata, "sigma")[draw_index]
    effects = {
        name: posterior_array(idata, f"u_{name}")[draw_index]
        for name in ("config", "site", "background", "generator", "phase")
    }
    seed_delta = int(hashlib.sha256(fit["fingerprint"].encode()).hexdigest()[:8], 16)
    rng = np.random.default_rng(SEED + seed_delta)

    frame = frame.copy()
    frame["frequency_key"] = (
        frame["model"].astype(str)
        + "|"
        + frame["f_lock"].map(lambda value: f"{value:.6f}Hz")
    )
    strata = []
    for stratum_type, column in (
        ("config", "model"),
        ("generator", "generator"),
        ("family", "family"),
        ("frequency", "frequency_key"),
    ):
        for level, raw_positions in frame.groupby(column, sort=True).indices.items():
            raw_positions = np.asarray(raw_positions, dtype=int)
            positions = raw_positions
            if len(positions) > max_obs_per_stratum:
                positions = positions[
                    np.linspace(0, len(positions) - 1, max_obs_per_stratum, dtype=int)
                ]
            mu = beta[:, None]
            for name in ("config", "site", "background", "generator", "phase"):
                mu = mu + effects[name][:, codes[name][positions]]
            if fit["likelihood"] == "student":
                noise = stats.t.rvs(NU, size=mu.shape, random_state=rng)
            else:
                noise = rng.normal(size=mu.shape)
            replicated = mu + sigma[:, None] * noise
            rep_mean = replicated.mean(axis=1)
            observed = float(y[positions].mean())
            lo, hi = np.quantile(rep_mean, [0.025, 0.975])
            strata.append({
                "fit": fit["label"], "stratum_type": stratum_type,
                "stratum": str(level), "n": len(raw_positions),
                "ppc_n": len(positions), "observed_mean": observed,
                "rep_lo": float(lo), "rep_hi": float(hi),
                "covered": bool(lo <= observed <= hi),
            })
    return pd.DataFrame(strata)

In [ ]:
def diagnostic_row(fit: dict, ppc: pd.DataFrame) -> dict:
    idata = fit["idata"]
    monitored = [
        name for name in idata.posterior.data_vars
        if name in {"beta_bar", "sigma"} or name.startswith("sigma_") or name.startswith("z_")
    ]
    def flatten_dataset(dataset) -> np.ndarray:
        # Do not call Dataset.to_array(): variables have different group dimensions,
        # and xarray would broadcast them over a catastrophic Cartesian product in FULL.
        arrays = [
            np.asarray(dataset[name].values, dtype=float).ravel()
            for name in dataset.data_vars
        ]
        return np.concatenate(arrays) if arrays else np.asarray([], dtype=float)

    rhat = flatten_dataset(az.rhat(idata, var_names=monitored, method="rank"))
    ess_bulk = flatten_dataset(az.ess(idata, var_names=monitored, method="bulk"))
    ess_tail = flatten_dataset(az.ess(idata, var_names=monitored, method="tail"))
    divergences = int(idata.sample_stats["diverging"].values.sum())
    bfmi = np.asarray(az.bfmi(idata), dtype=float)
    ppc_coverage_by_type = (
        ppc.groupby("stratum_type", sort=True)["covered"].mean().astype(float)
    )
    ppc_coverage = float(ppc_coverage_by_type.min())
    ppc_coverage_overall = float(ppc["covered"].mean())

    max_rhat = float(np.nanmax(rhat))
    min_ess_bulk = float(np.nanmin(ess_bulk))
    min_ess_tail = float(np.nanmin(ess_tail))
    min_bfmi = float(np.nanmin(bfmi))
    ok = bool(
        np.isfinite([max_rhat, min_ess_bulk, min_ess_tail, min_bfmi, ppc_coverage]).all()
        and max_rhat < GATE["max_rhat"]
        and min_ess_bulk > GATE["min_ess"]
        and min_ess_tail > GATE["min_ess"]
        and divergences <= GATE["max_divergences"]
        and min_bfmi >= GATE["min_bfmi"]
        and ppc_coverage >= GATE["min_ppc"]
    )
    return {
        "fit": fit["label"], "max_rhat": max_rhat,
        "min_ess_bulk": min_ess_bulk, "min_ess_tail": min_ess_tail,
        "divergences": divergences, "min_bfmi": min_bfmi,
        "ppc_coverage_min_type": ppc_coverage,
        "ppc_coverage_overall": ppc_coverage_overall,
        "ppc_coverage_by_type": json.dumps(
            ppc_coverage_by_type.round(6).to_dict(), sort_keys=True
        ),
        "gate_pass": ok,
    }

def hdi_interval(values: np.ndarray, probability: float = 0.95) -> tuple[float, float]:
    interval = np.asarray(az.hdi(np.asarray(values), hdi_prob=probability))
    return float(interval[0]), float(interval[1])

def posterior_summary(fit: dict) -> dict:
    beta = posterior_array(fit["idata"], "beta_bar")
    ratio = np.exp(beta)
    beta_lo, beta_hi = hdi_interval(beta)
    ratio_lo, ratio_hi = hdi_interval(ratio)
    paired_difference = fit["label"].startswith("paired_difference")
    return {
        "fit": fit["label"],
        "estimand": (
            "fixed-minus-adaptive contrast shift"
            if paired_difference else "lock recovery contrast"
        ),
        "beta_median": float(np.median(beta)),
        "beta_hdi_lo": beta_lo, "beta_hdi_hi": beta_hi,
        "ratio_median": float(np.median(ratio)),
        "ratio_hdi_lo": ratio_lo, "ratio_hdi_hi": ratio_hi,
        "p_attenuation_20": (
            np.nan if paired_difference else float(np.mean(beta < ATTENUATION_20))
        ),
        "p_any_attenuation": (
            np.nan if paired_difference else float(np.mean(beta < 0))
        ),
        "p_negative_shift": (
            float(np.mean(beta < 0)) if paired_difference else np.nan
        ),
        "p_in_rope": float(np.mean(np.abs(beta) < ROPE_LOG)),
    }

## Data

`FULL` writes resumable per-geometry shards and a manifest through
`collect_paired_offsets`. The manifest freezes geometry, seeds, offsets, excluded sites,
experimental constants, code hashes and resolved checkpoint identities. Set
`H1_ARTIFACT_ROOT` to a mounted Drive directory on Colab if artifacts must survive a runtime
disconnect. `SYNTHETIC_SMOKE` instead creates a deterministic non-reportable table and never
loads Chronos.

The wide table has one row per matched lock/background/phase and stores the common `R_lock`
plus all four controls. The two offset fits are kept separate; the paired-difference response
is \(d_{fixed}-d_{adaptive}\), which cancels the shared lock term exactly.

In [ ]:
if SYNTHETIC_SMOKE:
    paired = synthetic_paired_data(
        CONFIG,
        models=FROZEN_BAYES_MODELS[:2],
        n_bg=2,
        n_phase=2,
    )
else:
    paired = collect_paired_offsets(
        DATA_DIR,
        config=CONFIG,
        models=FROZEN_BAYES_MODELS,
        force=FORCE_COLLECTION,
    )

validation = validate_paired_data(
    paired,
    CONFIG,
    require_all_models=FULL,
)
display(pd.Series(validation, name="value").to_frame())

retained = {
    (pl.model_tag(P, S), round(float(frequency), 6))
    for P, S in FROZEN_BAYES_MODELS
    for frequency in select_common_sites(P, S, CONFIG)["f_lock"]
}
available = {
    (pl.model_tag(P, S), round(float(frequency), 6))
    for P, S in FROZEN_BAYES_MODELS
    for frequency in pl.f_lock(P, S)
}
excluded_sites = available - retained
expected_exclusions = {
    ("p32-s15", 238.933333),
    ("p32-s15", 240.0),
}
if excluded_sites != expected_exclusions:
    raise AssertionError(f"unexpected common-support exclusions: {excluded_sites}")
if len(retained) != 271:
    raise AssertionError(f"expected 271 common sites, found {len(retained)}")

if SYNTHETIC_SMOKE:
    smoke_counts = {
        "config": paired["model"].nunique(),
        "generator": paired["generator"].nunique(),
        "background": paired[["generator", "bg_id"]].drop_duplicates().shape[0],
        "site": paired[["model", "f_lock"]].drop_duplicates().shape[0],
        "phase": paired["phase_idx"].nunique(),
    }
    if min(smoke_counts.values()) < 2:
        raise AssertionError(f"synthetic smoke is too small: {smoke_counts}")
    if paired["reportable"].astype(bool).any():
        raise AssertionError("synthetic rows must be non-reportable")
else:
    if len(paired) != 487_400:
        raise AssertionError(f"expected 487,400 paired rows, found {len(paired):,}")
    if not paired["reportable"].astype(bool).all():
        raise AssertionError("FULL rows must be marked reportable")

data_hash_columns = [
    "pair_key", "model", "generator", "bg_id", "f_lock", "phase_idx",
    "d_fixed", "d_adaptive", "live_fixed", "live_adaptive",
]
data_hash = pd.util.hash_pandas_object(
    paired[data_hash_columns].sort_values("pair_key"), index=False
).values.tobytes()
DATA_FINGERPRINT = hashlib.sha256(
    DATA_DESIGN_FINGERPRINT.encode() + data_hash
).hexdigest()[:16]
REPORTABLE_DATA = bool(paired["reportable"].astype(bool).all())
print(f"data fingerprint: {DATA_FINGERPRINT}")
print(f"paired rows: {len(paired):,}")
print(f"common sites: {len(retained)} | excluded: {sorted(excluded_sites)}")

In [ ]:
long_contrasts = to_long_contrasts(paired)
if set(long_contrasts["offset_kind"].unique()) != {"fixed", "adaptive"}:
    raise AssertionError("long contrasts must contain fixed and adaptive rows")

fixed_all = long_contrasts[long_contrasts["offset_kind"] == "fixed"].copy()
adaptive_all = long_contrasts[long_contrasts["offset_kind"] == "adaptive"].copy()
fixed_live = fixed_all[fixed_all["live"].astype(bool)].copy()
adaptive_live = adaptive_all[adaptive_all["live"].astype(bool)].copy()

if set(fixed_all["pair_key"]) != set(adaptive_all["pair_key"]):
    raise AssertionError("fixed/adaptive pairing was lost")
if fixed_all["pair_key"].duplicated().any() or adaptive_all["pair_key"].duplicated().any():
    raise AssertionError("each arm must contain exactly one row per pair_key")
if fixed_live.empty or adaptive_live.empty:
    raise AssertionError("the live filter removed an entire offset arm")

meta_columns = [
    column for column in (
        "pair_key", "data_origin", "reportable", "model", "P", "S", "overlap",
        "generator", "bg_id", "f_lock", "family", "cpp", "phase_idx", "phase",
    ) if column in paired.columns
]
paired_difference_all = paired[meta_columns].copy()
paired_difference_all["offset_kind"] = "fixed_minus_adaptive"
paired_difference_all["delta_hz"] = np.nan
paired_difference_all["d"] = paired["d_fixed"] - paired["d_adaptive"]
paired_difference_all["live"] = (
    paired["live_fixed"].astype(bool) & paired["live_adaptive"].astype(bool)
)
paired_difference_live = paired_difference_all[
    paired_difference_all["live"].astype(bool)
].copy()

FIT_FRAMES = {
    "fixed_live": fixed_live,
    "fixed_all": fixed_all,
    "adaptive_live": adaptive_live,
    "adaptive_all": adaptive_all,
    "paired_difference_live": paired_difference_live,
    "paired_difference_all": paired_difference_all,
}
display(pd.DataFrame([
    {
        "dataset": label, "rows": len(frame),
        "configs": frame["model"].nunique(),
        "sites": frame[["model", "f_lock"]].drop_duplicates().shape[0],
    }
    for label, frame in FIT_FRAMES.items()
]))

In [ ]:
overview_rows = []
for kind, arm in long_contrasts.groupby("offset_kind", sort=True):
    overview_rows.append({
        "offset_kind": kind,
        "rows": len(arm),
        "configs": arm["model"].nunique(),
        "sites": arm[["model", "f_lock"]].drop_duplicates().shape[0],
        "live_fraction": arm["live"].mean(),
        "delta_min": arm["delta_hz"].min(),
        "delta_median": arm["delta_hz"].median(),
        "delta_max": arm["delta_hz"].max(),
    })
overview = pd.DataFrame(overview_rows).set_index("offset_kind")
display(overview.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for kind, color in (("fixed", "#1565c0"), ("adaptive", "#ef6c00")):
    values = long_contrasts.loc[long_contrasts["offset_kind"] == kind, "d"].dropna()
    if len(values) > 20000:
        values = values.sample(20000, random_state=SEED)
    axes[0].hist(values, bins=70, range=(-6, 6), density=True,
                 histtype="step", linewidth=1.5, label=kind, color=color)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(xlabel="paired log contrast d", ylabel="density",
            title="Raw contrast by offset")
axes[0].legend()

config_difference = (
    paired.assign(difference=lambda x: x["d_fixed"] - x["d_adaptive"])
    .groupby("model")["difference"].mean().sort_values()
)
axes[1].barh(config_difference.index, config_difference.values, color="#5e35b1")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="mean d_fixed - d_adaptive",
            title="Paired offset shift")

filter_diagnostic = paired.groupby("model").agg(
    fixed_live_fraction=("live_fixed", "mean"),
    adaptive_live_fraction=("live_adaptive", "mean"),
)
y = np.arange(len(filter_diagnostic))
axes[2].barh(y - 0.18, filter_diagnostic["fixed_live_fraction"],
             height=0.34, label="fixed", color="#1565c0")
axes[2].barh(y + 0.18, filter_diagnostic["adaptive_live_fraction"],
             height=0.34, label="adaptive", color="#ef6c00")
axes[2].set_yticks(y, filter_diagnostic.index)
axes[2].set(xlim=(0, 1), xlabel="retained fraction",
            title="Response-conditioned live filter")
axes[2].legend(fontsize=8)
fig.tight_layout()
plt.show()

# The reportable tables are already sharded on disk and all model frames above are compact.
# Releasing the million-row presentation table and five-response wide table keeps FULL within
# Colab RAM while posterior checkpoints remain reloadable by fingerprint.
del long_contrasts, paired


In [ ]:
prior_rows = []
for scale in PRIOR_LADDER:
    draws = stats.t.rvs(NU, loc=0.0, scale=scale, size=50000,
                        random_state=np.random.default_rng(SEED + int(scale * 100)))
    ratio = np.exp(draws)
    ratio_lo, ratio_hi = np.quantile(ratio, [0.025, 0.975])
    prior_rows.append({
        "scale": scale,
        "P(beta < log(0.8))": np.mean(draws < ATTENUATION_20),
        "P(beta < 0)": np.mean(draws < 0),
        "ratio_2.5%": ratio_lo,
        "ratio_median": np.median(ratio),
        "ratio_97.5%": ratio_hi,
    })
PRIOR_TABLE = pd.DataFrame(prior_rows)
display(PRIOR_TABLE.round(3))

## Results

The cells below deliberately compute results from the current data. In smoke mode they validate plumbing only. In full mode, interpret no posterior before checking the diagnostics table and the final gate.

In [ ]:
FITS = {}
for label, frame in FIT_FRAMES.items():
    print(f"\n--- fitting {label}: n={len(frame):,} ---")
    FITS[label] = fit_one(label, frame)

POSTERIOR_TABLE = pd.DataFrame(
    posterior_summary(fit) for fit in FITS.values()
)
display(POSTERIOR_TABLE.round(4))

In [ ]:
PPC_TABLES = {}
diagnostic_rows = []
for label, fit in FITS.items():
    ppc = posterior_predictive_strata(fit)
    PPC_TABLES[label] = ppc
    diagnostic_rows.append(diagnostic_row(fit, ppc))

DIAGNOSTICS = pd.DataFrame(diagnostic_rows)
display(DIAGNOSTICS.round(3))
failed = DIAGNOSTICS.loc[~DIAGNOSTICS["gate_pass"], "fit"].tolist()
print("primary diagnostic gate:", "PASS" if not failed else f"FAIL -> {failed}")

In [ ]:
contrast_order = ["fixed_live", "fixed_all", "adaptive_live", "adaptive_all"]
contrast_plot = POSTERIOR_TABLE.set_index("fit").loc[contrast_order].reset_index()
difference_order = ["paired_difference_live", "paired_difference_all"]
difference_plot = POSTERIOR_TABLE.set_index("fit").loc[difference_order].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, table, title in (
    (axes[0], contrast_plot, "Lock contrast by offset and view"),
    (axes[1], difference_plot, "Paired shift: fixed minus adaptive"),
):
    y = np.arange(len(table))
    lower = table["beta_median"] - table["beta_hdi_lo"]
    upper = table["beta_hdi_hi"] - table["beta_median"]
    ax.errorbar(
        table["beta_median"], y,
        xerr=np.vstack([lower, upper]),
        fmt="o", color="#1565c0", ecolor="#64b5f6", capsize=3,
    )
    ax.axvline(0, color="black", linewidth=0.9)
    ax.set_yticks(y, table["fit"])
    ax.set_xlabel("population effect (95% HDI)")
    ax.set_title(title)
axes[0].axvline(ATTENUATION_20, color="crimson", linestyle="--",
                linewidth=1.0, label="20% attenuation")
axes[0].axvspan(-ROPE_LOG, ROPE_LOG, color="grey", alpha=0.12, label="ROPE")
axes[0].legend(fontsize=8)
axes[1].axvspan(-ROPE_LOG, ROPE_LOG, color="grey", alpha=0.12,
                label="practical-equivalence band")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
def config_forest(fit: dict, title: str) -> None:
    idata, prepared = fit["idata"], fit["prepared"]
    beta = posterior_array(idata, "beta_bar")
    config_effect = beta[:, None] + posterior_array(idata, "u_config")
    medians = np.median(config_effect, axis=0)
    intervals = np.asarray([hdi_interval(config_effect[:, i]) for i in range(config_effect.shape[1])])
    labels = prepared["levels"]["config"]

    order = np.argsort(medians)
    y = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(8.5, max(3.5, 0.32 * len(order))))
    ax.errorbar(
        medians[order], y,
        xerr=np.vstack([
            medians[order] - intervals[order, 0],
            intervals[order, 1] - medians[order],
        ]),
        fmt="o", color="#1565c0", ecolor="#90caf9", capsize=2,
    )
    ax.axvline(0, color="black", linewidth=0.9)
    ax.axvline(ATTENUATION_20, color="crimson", linestyle="--", linewidth=1.0)
    ax.set_yticks(y, np.asarray(labels)[order])
    ax.set_xlabel("beta_bar + u_config (95% HDI)")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()

config_forest(FITS["fixed_live"], "Fixed ±1 Hz primary live view: configuration effects")

In [ ]:
ppc_config = PPC_TABLES["fixed_live"].query("stratum_type == 'config'").copy()
ppc_config = ppc_config.sort_values("observed_mean").reset_index(drop=True)
y = np.arange(len(ppc_config))

fig, ax = plt.subplots(figsize=(8.5, max(3.5, 0.32 * len(ppc_config))))
ax.hlines(y, ppc_config["rep_lo"], ppc_config["rep_hi"],
          color="#90caf9", linewidth=5, label="95% replicated means")
ax.scatter(ppc_config["observed_mean"], y, color="crimson",
           zorder=3, label="observed sampled mean")
ax.set_yticks(y, ppc_config["stratum"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("mean paired log contrast d")
ax.set_title("Fixed ±1 Hz primary-live PPC by configuration")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
trace = FITS["fixed_live"]["idata"].posterior["beta_bar"].values
fig, ax = plt.subplots(figsize=(9, 3.2))
for chain in range(trace.shape[0]):
    ax.plot(trace[chain], linewidth=0.7, alpha=0.8, label=f"chain {chain + 1}")
ax.axhline(ATTENUATION_20, color="crimson", linestyle="--", linewidth=0.9)
ax.set(xlabel="draw", ylabel="beta_bar",
       title="Fixed ±1 Hz primary live fit: beta_bar trace")
ax.legend(ncol=min(4, trace.shape[0]), fontsize=8)
fig.tight_layout()
plt.show()

### Parameter recovery

Recovery uses the real fixed-offset design matrix but replaces its response with data generated from the same zero-sum hierarchy at known population effects. It checks implementation and identifiability under the assumed model; it does not excuse a poor PPC on real data.

In [ ]:
def simulated_recovery_frame(
    template: pd.DataFrame,
    beta_true: float,
    *,
    seed: int,
) -> pd.DataFrame:
    prepared = prepare_model_frame(template)
    rng = np.random.default_rng(seed)
    scale_by_effect = {
        "config": 0.12, "site": 0.18, "background": 0.08,
        "generator": 0.05, "phase": 0.06,
    }
    mu = np.full(len(prepared["y"]), beta_true, dtype=float)
    for name, effect_scale in scale_by_effect.items():
        basis = prepared["bases"][name]
        coefficients = rng.normal(0, effect_scale, basis.shape[1])
        effect = basis @ coefficients
        mu += effect[prepared["codes"][name]]

    simulated = prepared["frame"].copy()
    simulated["d"] = mu + 0.30 * stats.t.rvs(
        NU, size=len(mu), random_state=rng
    )
    simulated["live"] = True
    simulated["data_origin"] = "parameter_recovery"
    simulated["reportable"] = False
    return simulated

RECOVERY_TRUTHS = {
    "null": 0.0,
    "20pct_attenuation": ATTENUATION_20,
    "50pct_attenuation": float(np.log(0.5)),
}

In [ ]:
RECOVERY_FITS = {}
recovery_rows = []
recovery_diagnostics = []

for index, (scenario, truth) in enumerate(RECOVERY_TRUTHS.items()):
    simulated = simulated_recovery_frame(
        fixed_live, truth, seed=SEED + 1000 + index
    )
    fit = fit_one(
        f"recovery_{scenario}",
        simulated,
        settings=AUX_SETTINGS,
    )
    RECOVERY_FITS[scenario] = fit
    beta = posterior_array(fit["idata"], "beta_bar")
    lo, hi = hdi_interval(beta)
    ppc = posterior_predictive_strata(fit)
    diag = diagnostic_row(fit, ppc)
    recovery_diagnostics.append(diag)
    recovery_rows.append({
        "scenario": scenario, "true_beta": truth,
        "posterior_median": float(np.median(beta)),
        "hdi_lo": lo, "hdi_hi": hi,
        "covers_truth": bool(lo <= truth <= hi),
        "diagnostic_gate": diag["gate_pass"],
    })

RECOVERY_TABLE = pd.DataFrame(recovery_rows)
RECOVERY_DIAGNOSTICS = pd.DataFrame(recovery_diagnostics)
display(RECOVERY_TABLE.round(4))
display(RECOVERY_DIAGNOSTICS.round(3))

### Prior and likelihood sensitivity

The primary fixed-offset `live` view is refitted with Student-\(t_4\) prior scales 0.25, 0.5
and 1.0, plus a Normal likelihood at scale 0.5. The primary scale-0.5 Student fit is reused by
fingerprint. A decision-class change is reported as prior/likelihood sensitivity; convergence
alone is not treated as substantive robustness.

In [ ]:
SENSITIVITY_FITS = {"StudentT scale 0.5": FITS["fixed_live"]}
for scale in (0.25, 1.0):
    label = f"fixed_live_student_scale_{scale}"
    SENSITIVITY_FITS[f"StudentT scale {scale}"] = fit_one(
        label, fixed_live, prior_scale=scale,
        likelihood="student", settings=AUX_SETTINGS,
    )
SENSITIVITY_FITS["Normal likelihood scale 0.5"] = fit_one(
    "fixed_live_normal_scale_0.5", fixed_live,
    prior_scale=0.5, likelihood="normal", settings=AUX_SETTINGS,
)

sensitivity_rows = []
sensitivity_diagnostics = []
for variant, fit in SENSITIVITY_FITS.items():
    row = posterior_summary(fit)
    row["variant"] = variant
    sensitivity_rows.append(row)
    ppc = posterior_predictive_strata(fit)
    diag = diagnostic_row(fit, ppc)
    diag["variant"] = variant
    sensitivity_diagnostics.append(diag)

SENSITIVITY_TABLE = pd.DataFrame(sensitivity_rows)
SENSITIVITY_DIAGNOSTICS = pd.DataFrame(sensitivity_diagnostics)
display(SENSITIVITY_TABLE[
    ["variant", "beta_median", "beta_hdi_lo", "beta_hdi_hi",
     "ratio_median", "p_attenuation_20", "p_in_rope"]
].round(4))
display(SENSITIVITY_DIAGNOSTICS.round(3))

SENSITIVITY_SPREAD = float(
    SENSITIVITY_TABLE["p_attenuation_20"].max()
    - SENSITIVITY_TABLE["p_attenuation_20"].min()
)
print(f"posterior-probability spread across sensitivity fits: {SENSITIVITY_SPREAD:.3f}")

In [ ]:
sensitivity_plot = SENSITIVITY_TABLE.set_index("variant").loc[list(SENSITIVITY_FITS)].reset_index()
y = np.arange(len(sensitivity_plot))
lower = sensitivity_plot["beta_median"] - sensitivity_plot["beta_hdi_lo"]
upper = sensitivity_plot["beta_hdi_hi"] - sensitivity_plot["beta_median"]

fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.errorbar(
    sensitivity_plot["beta_median"], y,
    xerr=np.vstack([lower, upper]),
    fmt="o", color="#5e35b1", ecolor="#b39ddb", capsize=3,
)
ax.axvline(0, color="black", linewidth=0.9)
ax.axvline(ATTENUATION_20, color="crimson", linestyle="--", linewidth=1.0)
ax.set_yticks(y, sensitivity_plot["variant"])
ax.set_xlabel("beta_bar (95% HDI)")
ax.set_title("Fixed ±1 Hz prior/likelihood sensitivity")
fig.tight_layout()
plt.show()

In [ ]:
primary_diagnostics_ok = bool(DIAGNOSTICS["gate_pass"].all())
recovery_coverage_ok = bool(RECOVERY_TABLE["covers_truth"].all())
recovery_diagnostics_ok = bool(RECOVERY_DIAGNOSTICS["gate_pass"].all())
sensitivity_diagnostics_ok = bool(SENSITIVITY_DIAGNOSTICS["gate_pass"].all())
reportable_data = REPORTABLE_DATA

def decision_class(row: pd.Series) -> str:
    if float(row["p_attenuation_20"]) >= 0.95:
        return "SUPPORTED"
    if float(row["p_in_rope"]) >= 0.95:
        return "PRACTICALLY ABSENT"
    return "INCONCLUSIVE"

posterior_index = POSTERIOR_TABLE.set_index("fit")
primary_row = posterior_index.loc["fixed_live"]
complete_row = posterior_index.loc["fixed_all"]
primary_class = decision_class(primary_row)
complete_class = decision_class(complete_row)
filter_robust = primary_class == complete_class

sensitivity_classes = {
    variant: decision_class(row)
    for variant, row in SENSITIVITY_TABLE.set_index("variant").iterrows()
}
sensitivity_robust = len(set(sensitivity_classes.values())) == 1

GATE_TABLE = pd.DataFrame([
    {"requirement": "FULL mode", "pass": FULL},
    {"requirement": "all data marked reportable", "pass": reportable_data},
    {"requirement": "fixed/adaptive/difference diagnostics", "pass": primary_diagnostics_ok},
    {"requirement": "parameter-recovery coverage", "pass": recovery_coverage_ok},
    {"requirement": "parameter-recovery diagnostics", "pass": recovery_diagnostics_ok},
    {"requirement": "sensitivity diagnostics", "pass": sensitivity_diagnostics_ok},
])
ANALYSIS_READY = bool(GATE_TABLE["pass"].all())

if SYNTHETIC_SMOKE:
    VERDICT = "NON-REPORTABLE SYNTHETIC SMOKE"
elif not ANALYSIS_READY:
    VERDICT = "BLOCKED: one or more validation gates failed"
elif not filter_robust:
    VERDICT = "INCONCLUSIVE: NOT ROBUST TO THE LIVE FILTER"
elif not sensitivity_robust:
    VERDICT = "INCONCLUSIVE: PRIOR/LIKELIHOOD SENSITIVE"
else:
    VERDICT = primary_class

VERDICT_RECORD = {
    "verdict": VERDICT,
    "run_mode": RUN_MODE,
    "reportable": bool(FULL and ANALYSIS_READY),
    "config_fingerprint": CONFIG_FINGERPRINT,
    "data_design_fingerprint": DATA_DESIGN_FINGERPRINT,
    "data_fingerprint": DATA_FINGERPRINT,
    "model_code_fingerprint": MODEL_CODE_FINGERPRINT,
    "nuts_sampler": NUTS_SAMPLER,
    "runtime_versions": runtime_versions,
    "primary_fit": "fixed_live",
    "primary_decision": primary_class,
    "complete_view_decision": complete_class,
    "robust_to_live_filter": filter_robust,
    "robust_to_prior_likelihood": sensitivity_robust,
    "sensitivity_decisions": sensitivity_classes,
    "p_attenuation_20": float(primary_row["p_attenuation_20"]),
    "p_in_rope": float(primary_row["p_in_rope"]),
    "decision_threshold": 0.95,
    "attenuation_threshold": ATTENUATION_20,
    "rope_half_width": ROPE_LOG,
    "sensitivity_probability_spread": SENSITIVITY_SPREAD,
}
display(GATE_TABLE)
display(pd.Series(VERDICT_RECORD, name="value").to_frame())

In [ ]:
POSTERIOR_TABLE.to_csv(RESULT_DIR / f"posterior__{DATA_FINGERPRINT}.csv", index=False)
DIAGNOSTICS.to_csv(RESULT_DIR / f"diagnostics__{DATA_FINGERPRINT}.csv", index=False)
RECOVERY_TABLE.to_csv(RESULT_DIR / f"recovery__{DATA_FINGERPRINT}.csv", index=False)
SENSITIVITY_TABLE.to_csv(RESULT_DIR / f"sensitivity__{DATA_FINGERPRINT}.csv", index=False)
with (RESULT_DIR / f"verdict__{DATA_FINGERPRINT}.json").open("w", encoding="utf-8") as handle:
    json.dump(VERDICT_RECORD, handle, indent=2, sort_keys=True)

print(f"saved fingerprinted result tables to {RESULT_DIR}")

## Takeaways

- Fixed and adaptive estimates answer different localization questions and are reported side
  by side; the paired difference quantifies the change of estimand without duplicating `R_lock`.
- `fixed_live` is primary by design, while `fixed_all` is mandatory. Because `live` conditions
  on observed recovery, disagreement forces the explicit verdict `NOT ROBUST TO THE LIVE FILTER`.
- Practical absence is a positive ROPE claim, not the complement of 20% attenuation support.
- A near-zero population effect is not uniform absence: inspect the geometry forest and PPC.
- `BLOCKED` means diagnostics are not reportable; it is not permission to quote the posterior.
- `SYNTHETIC_SMOKE` is always non-reportable. A full Colab run should use a GPU runtime and
  `H1_NUTS_SAMPLER=numpyro`; all 21 retrained checkpoints must be accessible.

From PowerShell, execute the reportable workflow with the locked Python 3.11 environment:

```powershell
$env:H1_RUN_MODE = 'FULL'
$env:H1_NUTS_SAMPLER = 'numpyro'
uv run --locked --with jupyter --with numpyro jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=-1 chronos/bayesian/h1_fixed_offset_analysis.ipynb
```

To persist Colab shards and posterior checkpoints, point `H1_ARTIFACT_ROOT` at a mounted Drive
directory before execution. Outputs remain isolated under `h1_fixed_offset/` and carry
configuration and data fingerprints.